In [ ]:
import os
import gc
import sys
import glob
import numpy as np
import pandas as pd
import netCDF4 as nc
from datetime import datetime, timedelta
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib import colors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import multiprocessing as mp

In [ ]:
# To use PLUMBER2_GPP_common_utils, change directory to where it exists
os.chdir('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2')
from PLUMBER2_GPP_common_utils import *

In [ ]:
# Path of PLUMBER 2 dataset
PLUMBER2_path      = "/srv/ccrc/LandAP/z5218916/data/PLUMBER2/"
PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"

site_names, IGBP_types, clim_types, model_names = load_default_list()

remove_site        = get_removed_site_names()

models_calc_LAI   = ['ORC2_r6593','ORC2_r6593_CO2','ORC3_r7245_NEE','ORC3_r8120','GFDL','SDGVM','QUINCY','NoahMPv401']
model_LAI_names   = {'ORC2_r6593':'lai','ORC2_r6593_CO2':'lai','ORC3_r7245_NEE':'lai','ORC3_r8120':'lai',
                     'GFDL':'lai', 'SDGVM':'lai','QUINCY':'LAI','NoahMPv401':'LAI'} #

# Calculate remaining sites
set_site_names      = set(site_names)
set_remove_site     = set(remove_site)
remain_sites        = set_site_names - set_remove_site
remain_sites        = list(remain_sites)

In [ ]:
model_colors ={0:'red', 1: 'darkorange',2:'orange',3:'gold',4:'yellowgreen',5:'green',6:'mediumseagreen',
               7:'lime',8:'aquamarine',9:'cyan',10:'dodgerblue',11:'blue',12:'darkolivegreen',
               13:'forestgreen',14:'lime',15:'gold', 16:'orange',17:'pink',18:'pink',19:'red',20:'deeppink',
               21:'mediumorchid',22: 'darkviolet',}

IGBP_colors  = set_IGBP_colors()
clim_colors  = set_clim_colors()

<h3 style="color:blue;">Modelled vs obsed NEE</h3>  

<h4 style="color:green;">Monthly NEE regression with annual NEE</h4>  

In [ ]:
def plot_modelled_vs_observed(var_name,model_in, per_LAI):
    
    if per_LAI:
        df_obs   = pd.read_csv(f'./txt/{var_name}_annual_mean_per_LAI_obs.csv')
        df_model = pd.read_csv(f'./txt/{var_name}_annual_mean_per_LAI_{model_in}.csv')
    else:
        df_obs   = pd.read_csv(f'./txt/{var_name}_annual_mean_obs.csv')
        df_model = pd.read_csv(f'./txt/{var_name}_annual_mean_{model_in}.csv')
    print(df_obs)
    var_max = np.nanmax((df_model[var_name], df_obs[var_name]))
    var_min = np.nanmin((df_model[var_name], df_obs[var_name]))
    print(var_max,var_min)
    fig = plt.figure(figsize=(10, 5))
    ax  = plt.axes()
    
    # read site charaters
    df_char = pd.read_csv('./txt/site_character.csv')
    
    for site_name in site_names:
        mask_model = df_model['site_name'] == site_name
        mask_obs   = df_obs['site_name']   == site_name
        mask_char  = df_char['site_name']  == site_name
        
        obs        = df_obs.loc[mask_obs,'NEE'].values
        model      = df_model.loc[mask_model,'NEE'].values
        IGBP_type  = df_char.loc[mask_char,'IGBP'].values
        clim_type  = df_char.loc[mask_char,'clim_type'].values
        ax.scatter(obs, model,color=clim_colors[clim_type[0]])
        
    # ax.plot(np.linspace(var_min,var_max,100),np.linspace(var_min,var_max,100),color='black')
    plt.show()
    

In [ ]:
plot_modelled_vs_observed('NEE','CABLE', per_LAI=True)